In [76]:
%pip install langchain langchain-community langchain-openai chromadb pypdf scikit-learn


  Obtaining dependency information for langchain from https://files.pythonhosted.org/packages/30/03/8cc027f9d297473de94f134546dae2bfca255161ebbda988522fc509e1d5/langchain-1.4.3-py3-none-any.whl.metadata
  Obtaining dependency information for langchain-community from https://files.pythonhosted.org/packages/8f/39/5d97e42a3e95dc2a6d71b2f902a3fae71786131e11d01bddb604accb0ebe/langchain_community-0.4.2-py3-none-any.whl.metadata
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Obtaining dependency information for langchain-openai from https://files.pythonhosted.org/packages/18/d9/b3db61c6c15766c999194a9cba0b9c5035567b7e0dd1c1a94a6d76448cd6/langchain_openai-1.6.7-py3-none-any.whl.metadata
  Obtaining dependency information for chromadb from https://files.pythonhosted.org/packages/eb/ce/0f7be6e5d0feafa2cda54b12e6542afeea7dea89d2d411e14da90f8abb96/chromadb-1.5.9-cp39-abi3-win_amd64.whl.metadata
  Using cached chromadb-1.5.9-cp39-abi3-win_amd64.whl.metadata (5.1 kB)
 


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from langchain_chroma import Chroma
import os
import chromadb
import pathlib
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

C:\Users\Open User\AppData\Local\Temp\ipykernel_18096\3558468604.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
pdf_docs = PyPDFLoader("Lagos_Transit.pdf").load()


print("Pages loaded from the PDF :", len(pdf_docs))


print()
print("First 300 characters of the PDF:")
print(pdf_docs[0].page_content[:300])

print()


Pages loaded from the PDF : 47

First 300 characters of the PDF:
Lagos Commuter & Transit Survey - 333 responses
Page 1
Lagos Commuter and Transit Survey - Retrieval-Optimised Knowledge
Base
This document summarises a survey of 333 Lagos commuters. Each section is written so that a single paragraph can be
understood on its own. Currency is the Nigerian Naira, wri



In [3]:
# The same splitter is used for both sources
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

# The PDF pages are already documents, so we split the documents
pdf_chunks = splitter.split_documents(pdf_docs)

# The TXT is one long plain string, so we create documents from it
#txt_chunks = splitter.create_documents([txt_text])

print("Chunks from the PDF :", len(pdf_chunks))
#print("Chunks from the TXT :", len(txt_chunks))


# ---------------------------------------------------------------
# CHOOSE WHICH SOURCE TO EMBED
# Uncomment the line you want and comment out the others.
# ---------------------------------------------------------------
#chosen_chunks = pdf_chunks + txt_chunks     # use both together
chosen_chunks = pdf_chunks                # use only the PDF
# chosen_chunks = txt_chunks                # use only the text file


# Chroma only needs the plain text of each chunk
documents = [chunk.page_content for chunk in chosen_chunks]

print()
print("Documents ready to embed:", len(documents))

Chunks from the PDF : 345

Documents ready to embed: 345


In [4]:




print(f"Total Documents to be embedded {len(documents)}\n")

Total Documents to be embedded 345



In [5]:
load_dotenv()

# ---------------------------------------------------------------
# PROVIDER CONFIG
# The API keys are secret, so they live in the .env file.
# Everything else is written out here so you can see it.
#
# NOTE: the chat model and the embedding model run on TWO DIFFERENT
# servers. Each one needs its own base URL and its own key.
# Do not mix them up.
# ---------------------------------------------------------------

# --- Chat model ---
api_key         = os.getenv("OPENAI_API_KEY")
base_url        = "https://api.groq.com/openai/v1"
chat_model_name = "openai/gpt-oss-20b"

# --- Embedding model (different server, different key) ---
embedding_api_key    = os.getenv("EMBEDDING_API_KEY")
embedding_base_url   = "https://qwen-embed.publicaai.com/v1"
embedding_model_name = "Qwen/Qwen3-Embedding-0.6B"

print("Chat base URL     :", base_url)
print("Chat model        :", chat_model_name)
print("Embed base URL    :", embedding_base_url)
print("Embedding model   :", embedding_model_name)

Chat base URL     : https://api.groq.com/openai/v1
Chat model        : openai/gpt-oss-20b
Embed base URL    : https://qwen-embed.publicaai.com/v1
Embedding model   : Qwen/Qwen3-Embedding-0.6B


In [6]:
chroma_path = "lagos_transit_store"


In [7]:
# Initialize the embedding model
embeddings = OpenAIEmbeddings(
    model=embedding_model_name,
    api_key=embedding_api_key,
    base_url=embedding_base_url

)

In [8]:
# Initialize ChromaDB
lagos_transit_db= Chroma(
    collection_name="lagos_transit",
    embedding_function=embeddings,
    persist_directory=chroma_path
)

In [9]:
# Run this ONLY if the cell above restarts your kernel.
# This does not use Chroma. It saves into a folder, so you embed only once.
import os
from langchain_community.vectorstores import SKLearnVectorStore

os.makedirs("lagos_transit_store", exist_ok=True)
store_path = "lagos_transit_store/lagos_transit_index.json"

lagos_transit_db = SKLearnVectorStore.from_texts(
    documents,
    embedding=embeddings,
    persist_path=store_path,
    serializer="json",
)
lagos_transit_db.persist()

print("Saved", len(documents), "documents to", store_path)

Saved 345 documents to lagos_transit_store/lagos_transit_index.json


In [10]:
# Maximal Marginal Relevance (MMR) for diverse and relevant results.
lagos_transit_retriever = lagos_transit_db.as_retriever(search_type="mmr")

In [11]:
lagos_transit_retriever = lagos_transit_db.as_retriever(search_type="mmr", search_kwargs={'k': 4, 'fetch_k': 10})

In [12]:
question = "How long does a business name reservation last in Nigeria?"
lagos_transit_retriever.invoke(question)             

[Document(metadata={'id': 'fb980c49-0bbe-4767-9bf4-234a86ec184b'}, page_content='minutes.\n[Record 191] Trip from Igbogbo (Ikorodu LGA) to Garage (Other / Border Area LGA), on a Weekend (Saturday - Sunday),\ndeparting in the Mid-Morning / Off-Peak (8:00 AM - 11:59 AM) period, by Keke Marwa (Tricycle). Fare paid: NGN 500. Travel time:'),
 Document(metadata={'id': '01e03c43-ca10-4ff5-811f-6225d2c3e592'}, page_content='hours. Main problem: Bad road especially when it raining.\n[Record 162] Trip from Iyana Ipaja (Alimosho LGA) to Ikeja (Ikeja LGA), on a Weekday (Monday - Friday), departing in the Early\nMorning Rush (5:00 AM - 7:59 AM) period, by Danfo (Yellow bus). Fare paid: NGN 3,000. Travel time: 2 to 3 hours. Main problem:\nTraffic jam And high price of transportation once it is evening time.\n[Record 163] Trip from Garage ikorodu (Ikorodu LGA) to Mile 2 (Amuwo-Odofin LGA), on a Weekday (Monday - Friday), departing\nin the Early Morning Rush (5:00 AM - 7:59 AM) period, by BRT (Blue/Re

In [13]:
#initialize the chatmodel
chatmodel = ChatOpenAI(
    api_key=api_key,
    base_url=base_url,
    model=chat_model_name,
    temperature=0
)

In [14]:
#using from template method
from langchain_core.prompts import ChatPromptTemplate

# Custom RAG Prompt Template for Lagos Transport Intelligence Platform
prompt = ChatPromptTemplate.from_template(
    """You are an expert AI Transit Consultant and Route Assistant for Lagos State, Nigeria.
You will be provided with context containing factual commuter survey data, route costs, travel durations, peak traffic times, and transit challenges across all 20 Local Government Areas (LGAs) in Lagos.

Context:
{context}

User Question:
{question}

Instructions:
1. Provide an accurate, clear, and actionable response based strictly on the provided transit context.
2. If the user asks about fares, travel times, or route recommendations, extract precise numbers (in Nigerian Naira ₦) and estimates from the context.
3. Keep the tone professional, concise, and helpful to a Lagos commuter or transit planner.
4. If the context does not contain enough information to fully answer the question, state what is known from the context and gently advise the user to refine their query.
5. the transport fare for to is the same thing as fro e.g transport from ikeja to ikorodu is the same as ikorodu to ikeja
Answer:"""
)


In [15]:
question = "What is the average fare and travel time from ikorodu to Ikeja during evening rush?"

#retrieve the document
get_doc = lagos_transit_retriever.invoke(question)
# Prepare the input for the chain
input = {"context": get_doc, "question": question}

# Create the chain
chain = prompt | chatmodel | StrOutputParser()

answer = chain.invoke(input)
print(answer)

**Ikorodu → Ikeja (Evening Rush)**  

| Metric | Value | Source |
|--------|-------|--------|
| **Average fare** | ₦1,233 | 3 survey responses for the Ikeja → Ikorodu evening‑rush leg (symmetrical). |
| **Estimated travel time** | ~45 minutes | Most common time band 30–60 min; average calculated from band mid‑points. |

*Note:* The survey treats the fare and travel time as symmetric, so the figures for Ikeja → Ikorodu apply to Ikorodu → Ikeja. If you need more granular data (e.g., by specific departure times or vehicle type), let me know!
